### Read the Bronze orders table

In [0]:
from pyspark.sql import functions as F

BRONZE_TABLE = "olist_mlops.bronze.orders"
SILVER_TABLE = "olist_mlops.silver.orders_clean"

bronze_orders = spark.table(BRONZE_TABLE)

print("Bronze row count:", bronze_orders.count())

bronze_orders.printSchema()

### Understand type conversion

In [0]:
timestamp_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

silver_orders = bronze_orders

for column in timestamp_columns:

    silver_orders = silver_orders.withColumn(
        column,
        F.expr(
            f"try_cast(`{column}` AS TIMESTAMP_NTZ)"
        )
    )

silver_orders = silver_orders.withColumn(
    "order_purchase_date",
    F.to_date("order_purchase_timestamp")
)

silver_orders.printSchema()

### Validate timestamp conversion

In [0]:
for column in timestamp_columns:

    invalid_count = bronze_orders.filter(
        F.col(column).isNotNull()
        & F.expr(
            f"try_cast(`{column}` AS TIMESTAMP_NTZ)"
        ).isNull()
    ).count()

    print(
        f"{column}: {invalid_count} invalid timestamps"
    )

    assert invalid_count == 0, (
        f"Invalid timestamps found in {column}"
    )

print("Timestamp conversion validation PASSED")

### Validate row counts and candidate primary keys

In [0]:
bronze_count = bronze_orders.count()
silver_count = silver_orders.count()

null_keys = silver_orders.filter(
    F.col("order_id").isNull()
).count()

duplicate_keys = (
    silver_orders
    .groupBy("order_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("Bronze rows:", bronze_count)
print("Silver rows:", silver_count)
print("Null order IDs:", null_keys)
print("Duplicate order IDs:", duplicate_keys)

assert bronze_count == silver_count == 99441
assert null_keys == 0
assert duplicate_keys == 0

print("Silver data validation PASSED")

### Write our first Silver Delta table

In [0]:
(
    silver_orders.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_TABLE)
)

print("Created:", SILVER_TABLE)

assert spark.table(SILVER_TABLE).count() == bronze_count

### Verify the new Silver table using SQL

In [0]:
%sql
DESCRIBE TABLE olist_mlops.silver.orders_clean;

In [0]:
%sql
-- Verify that the five timestamp columns have the TIMESTAMP_NTZ data type
SELECT
    order_id,
    order_status,
    order_purchase_timestamp,
    order_purchase_date,
    order_delivered_customer_date,
    order_estimated_delivery_date
FROM olist_mlops.silver.orders_clean
LIMIT 10;

### Investigate the delivery-delay target

In [0]:
%sql
SELECT
    COUNT(*) AS total_orders,
    SUM(
        CASE
            WHEN date_format(
                order_estimated_delivery_date,
                'HH:mm:ss'
            ) = '00:00:00'
            THEN 1 ELSE 0
        END
    ) AS estimated_at_midnight
FROM olist_mlops.silver.orders_clean;

In [0]:
%sql
-- compare our original timestamp-based label with the proposed date-based label:
SELECT
    COUNT(*) AS eligible_orders,

    SUM(
        CASE
            WHEN order_delivered_customer_date >
                 order_estimated_delivery_date
            THEN 1 ELSE 0
        END
    ) AS late_by_timestamp,

    SUM(
        CASE
            WHEN CAST(order_delivered_customer_date AS DATE) >
                 CAST(order_estimated_delivery_date AS DATE)
            THEN 1 ELSE 0
        END
    ) AS late_by_calendar_date

FROM olist_mlops.silver.orders_clean

WHERE order_status = 'delivered'
  AND order_delivered_customer_date IS NOT NULL
  AND order_estimated_delivery_date IS NOT NULL;

In [0]:
%sql

SELECT
    COUNT(*) AS total_orders,

    SUM(
        CASE
            WHEN date_format(
                order_estimated_delivery_date,
                'HH:mm:ss'
            ) = '00:00:00'
            THEN 1 ELSE 0
        END
    ) AS estimates_at_midnight,

    SUM(
        CASE
            WHEN date_format(
                order_estimated_delivery_date,
                'HH:mm:ss'
            ) <> '00:00:00'
            THEN 1 ELSE 0
        END
    ) AS estimates_not_at_midnight

FROM olist_mlops.silver.orders_clean;